# 财政动态分析 —— 中国省级税收数据挖掘与计量经济学分析

1. 数据读取与初步探索
2. 数据整合与统计描述
3. 数据处理与清洗
4. 相关性分析与回归建模
5. 聚类分析与区域分级
6. 地理空间可视化

---
## 一、实验概述

### 研究目标
- **变量相关性分析**：探究产业税收与居民人均消费支出、人均可支配收入之间的线性关联程度
- **区域税收分级**：基于税收数据特征，通过聚类分析对不同地区进行税收等级分组，揭示区域税收差异

### 全局设置

In [ ]:
# === 内置库 ===
import os
import warnings
from copy import deepcopy

# === 第三方库 ===
import numpy as np
import pandas as pd
import geopandas as gpd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from IPython.display import display, HTML

# === 统计与机器学习 ===
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.stattools import durbin_watson
from statsmodels.stats.diagnostic import acorr_breusch_godfrey, het_white
from statsmodels.stats.anova import anova_lm
from sklearn.cluster import KMeans

# === 全局显示设置 ===
pd.set_option('display.float_format', lambda x: f'{x:.2f}')

# === matplotlib 中文字体设置（SimHei 或系统可用中文字体）===
plt.rcParams.update({'font.family': 'SimHei', 'axes.unicode_minus': False})
# 如果 SimHei 不可用，尝试以下字体：
# plt.rcParams.update({'font.sans-serif': ['WenQuanYi Micro Hei', 'SimHei', 'Microsoft YaHei']})

# === 忽略无关警告 ===
warnings.filterwarnings("ignore", category=UserWarning, module=r"(openpyxl|sklearn)")

print("环境初始化完成。")

### 数据路径配置

论文使用的数据集包含三张表：
- **产业税收表**：含 TaxAmount, PrimaryInduTaxAmount, SecondInduTaxAmount, TertiaryInduTaxAmount 等
- **居民人均消费支出表**：含 ConsumExpense, CE_FoodTobaccoliquor, CE_Clothing, CE_Resident 等
- **人均可支配收入表**：含 PCDI, PCDI_WageIncome, PCDI_BusinessIncome, PCDI_PropertyIncome, PCDI_TransferIncome

In [ ]:
# 数据目录（相对于项目根目录）
DATA_PATH = os.path.join('..', 'data')

# 如果你已将数据放在其他位置，修改上面的路径
# 例如: DATA_PATH = r'/path/to/your/data'

print(f"数据目录: {os.path.abspath(DATA_PATH)}")

---
## 二、数据读取与初步查看

遍历数据目录，批量读取所有 Excel 文件，初步了解各数据集的字段构成与数据特征。

In [ ]:
# 读取数据目录下所有 Excel 文件（跳过临时文件）
es = [os.path.join(DATA_PATH, f) for f in os.listdir(DATA_PATH) if not f.startswith('~$')]
dfs = [pd.read_excel(f) for f in es if f]

print(f"共读取 {len(dfs)} 个数据文件:")
for i, (f, df) in enumerate(zip(es, dfs)):
    print(f"  文件 {i+1}: {os.path.basename(f)} — {df.shape[0]} 行 × {df.shape[1]} 列")

In [ ]:
# 自定义表格展示函数
def q(df_slice, justify="center"):
    """以 HTML 表格形式展示 DataFrame 切片。"""
    display(HTML(df_slice.to_html(justify=justify)))

# 查看每个表的前 3 行
for i in range(len(dfs)):
    df = dfs[i]
    print(f'=== 第 {i+1} 个表 {"%" * 60}')
    q(df[:3])

---
## 三、数据整合与统计描述

### 3.1 左外连接

以产业税收表为左表，通过左连接整合居民人均消费支出表和人均可支配收入表，构建综合数据集。

In [ ]:
# 左外连接: 税收表 ← 消费支出表 ← 可支配收入表
# 注: 论文中 dfs 列表顺序为 [消费支出表, 可支配收入表, 税收表]
# 如有不同，请根据实际文件顺序调整索引

df1 = pd.merge(dfs[-1], dfs[0], how='left')   # 税收 + 消费支出
dfm = pd.merge(df1, dfs[1], how='left')        # 再 + 可支配收入

print(f"合并后数据集: {dfm.shape[0]} 行 × {dfm.shape[1]} 列")

### 3.2 整体信息查看

In [ ]:
# 查看前几行
q(dfm.head())

# 查看数值列的基本统计信息
q(dfm.iloc[:, 3:].describe())

# 查看数据类型和缺失情况
dfm.info()

### 3.3 数据类型转换

数据列被读为 object 类型，需要转换为数值类型。

In [ ]:
# 将分类列以外的列转换为数值
for col in dfm.columns[3:]:
    dfm[col] = pd.to_numeric(dfm[col], errors='coerce')

print("数据类型转换完成。")
print(f"数值列: {dfm.select_dtypes(include=['float64', 'int64']).columns.tolist()}")

### 3.4 分类变量查看

In [ ]:
print("年份范围:", dfm['Sgnyea'].unique())
print("\n地区名称:", dfm['AreaName'].unique())

### 3.5 数值变量统计信息

自定义统计函数，计算各数值列的最小值、众数、中位数、偏度和峰度。

In [ ]:
def statistic_info(dfm, column_name: str) -> pd.DataFrame:
    """计算指定列的常用统计量。
    
    参数:
        dfm: 数据表
        column_name: 列名
    返回:
        包含统计量名称和值的 DataFrame
    """
    column_data = dfm[column_name]
    result_df = pd.DataFrame({
        '统计量': ['最小值', '最大众数', '众数个数', '中位数', '偏度', '峰度'],
        '值': [
            column_data.min(),
            column_data.mode().iloc[0] if not column_data.mode().empty else None,
            len(column_data.mode()),
            column_data.median(),
            column_data.skew(),
            column_data.kurtosis()
        ]
    }, index=[column_name] * 6)
    result_df.index.name = '列名'
    return result_df

# 以 PrimaryInduTaxAmount 为例
q(statistic_info(dfm, 'PrimaryInduTaxAmount'))

### 3.6 特征变量分布

In [ ]:
# 直方图查看各特征分布
dfm.hist(figsize=(16, 20), bins=50, xlabelsize=8, ylabelsize=8)
plt.tight_layout()
plt.show()

---
## 四、数据处理

### 4.1 剔除无关数据

In [ ]:
# 1. 删除与实验无关的列
dfm.drop(['AreaCode', 'StatisticalScope'], axis=1, inplace=True)

# 2. 删除地区名称为"中国"的行（汇总行，非省级数据）
dfm = dfm.drop(dfm[dfm['AreaName'] == '中国'].index).reset_index(drop=True)
print(f"剔除无关数据后: {dfm.shape[0]} 行 × {dfm.shape[1]} 列")

In [ ]:
# 3. 检查并删除包含负数的行
dfnum = dfm.select_dtypes(include=['float64', 'int64'])
for c in dfnum.columns:
    if (dfnum[c] < 0).any():
        fu = dfnum[dfnum[c] < 0].index
        print(f"列 '{c}' 包含负数，负数所在索引: {list(fu)}")
        print('-' * 50)
        dfm.drop(fu, inplace=True)

# 4. 填补 TaxAmount：该列是第一、二、三产业税收之和
ta = dfm.columns[3:6]  # PrimaryInduTaxAmount, SecondInduTaxAmount, TertiaryInduTaxAmount
valid_rows = dfm[ta].dropna().index
dfm.loc[valid_rows, 'TaxAmount'] = dfm.loc[valid_rows, ta].sum(axis=1)

print("数据清洗完成。")
dfm.info()

### 4.2 缺失值处理

先可视化缺失值分布，再删除含有缺失值的行。

In [ ]:
# 缺失值分布热力图
plt.figure(figsize=(10, 8))
sns.heatmap(dfm.isnull(), cbar=False, yticklabels=False, cmap='viridis')
plt.title('缺失值分布热图', fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
# 基于 PCDI_WageIncome 列删除含有缺失值的行
# 论文中该列缺失率约 50%，但数据量可观，直接删除
dfx = deepcopy(dfm)
dfx = dfx.dropna(subset=['PCDI_WageIncome'])

missing_values = dfx.isnull().sum()
if any(missing_values > 0):
    print("存在缺失值的列：")
    print(missing_values[missing_values > 0])
else:
    print("数据中没有缺失值。")

print(f"\n缺失值处理后: {dfx.shape[0]} 行 × {dfx.shape[1]} 列")

### 4.3 对数变换

对数值变量取对数，以压缩数据范围、削弱异常值影响、缓解右偏分布，使数据接近正态分布。

In [ ]:
# 对所有数值列进行对数变换
scol = dfx.select_dtypes(include=['float64', 'int64']).columns
dfx[scol] = np.log(dfx[scol])

# 查看变换后的统计信息
q(dfx.describe())

### 4.4 正态性验证

通过直方图、核密度估计（KDE）和 Shapiro-Wilk 检验，验证对数变换后数据是否近似正态分布。

In [ ]:
# i. 直方图
dfx.hist(figsize=(16, 20), bins=50, xlabelsize=8, ylabelsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# ii. 核密度估计图
dfx.select_dtypes(include=['int64', 'float64']).plot.kde(figsize=(10, 6), legend=False)
plt.title('Kernel Density Estimation Plot')
plt.xlabel('Value')
plt.ylabel('Density')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Variables', fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# iii. Shapiro-Wilk 正态性检验
# W 值越接近 1，数据越接近正态分布；p > 0.05 表示不拒绝正态分布原假设
d = {c: p for c in scol if stats.shapiro(dfx[c])[1] > 0.05}
if d:
    shapiro_df = pd.DataFrame(list(d.items()), columns=['列名', 'Shapiro_p值'])
    q(shapiro_df)
else:
    print("所有数值列均未拒绝正态分布原假设（p > 0.05）。")

---
## 五、相关性分析

### 5.1 Pearson 相关系数

数据已通过对数变换近似正态分布，可使用 Pearson 相关系数衡量变量间的线性关联。

In [ ]:
# 计算 Pearson 相关系数矩阵
corr_matrix = dfx.corr(numeric_only=True)

# 热力图可视化
plt.figure(figsize=(12, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Correlation with TaxAmount', fontsize=14)
plt.tight_layout()
plt.show()

### 5.2 多元线性回归

以总税收（TaxAmount）为因变量，居民消费支出和可支配收入各子项为自变量，构建 OLS 回归模型。

In [ ]:
# 构建回归公式: TaxAmount ~ 消费变量 + 收入变量
formula = f"{scol[0]} ~ " + " + ".join(scol[4:])
print(f"回归公式: {formula}\n")

# 使用 statsmodels 公式 API 拟合模型
model = smf.ols(formula=formula, data=dfx).fit()

print(f"=== 因变量: {scol[0]} ===")
print(model.summary())

### 5.3 ANOVA 分析

In [ ]:
# 方差分析表
anova_table = anova_lm(model, typ=1)
print("=== ANOVA 表 ===")
print(anova_table)

### 5.4 多重共线性检验（VIF）

方差膨胀因子（VIF）用于诊断自变量之间的多重共线性。VIF > 10 表示存在严重的多重共线性。

In [ ]:
def vif(df_exog, exog_name):
    """计算指定变量的方差膨胀因子（VIF）。
    
    VIF = 1 / (1 - R²)，其中 R² 是以该变量为因变量、其余变量为自变量时的决定系数。
    """
    exog_use = list(df_exog.columns)
    exog_use.remove(exog_name)
    model = smf.ols(f"{exog_name} ~ {' + '.join(exog_use)}", data=df_exog).fit()
    return 1 / (1 - model.rsquared)

# 计算各变量的 VIF 和容忍度
df_vif = pd.DataFrame({x: vif(dfx.iloc[:, 4:], x) for x in scol[4:]}, index=['VIF']).T
df_vif['tolerance'] = 1 / df_vif['VIF']

print("=== 多重共线性检验（VIF）===")
print(df_vif)

### 5.5 异方差检验

通过残差图和 White 检验判断回归模型是否存在异方差。

In [ ]:
# 获取预测值和残差
y_pred = model.predict()
residuals = model.resid

# i. 残差 vs 拟合值图
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].scatter(y_pred, residuals, alpha=0.5)
axes[0].axhline(y=0, color='r', linestyle='--')
axes[0].set_xlabel('拟合值')
axes[0].set_ylabel('残差')
axes[0].set_title('残差 vs 拟合值')

# ii. 残差 Q-Q 图
sm.qqplot(residuals, line='s', ax=axes[1])
axes[1].set_title('残差的 Q-Q 图')

# iii. 残差分布直方图
axes[2].hist(residuals, bins=30, density=True, alpha=0.7, edgecolor='black')
axes[2].axvline(x=0, color='r', linestyle='--')
axes[2].set_xlabel('残差')
axes[2].set_ylabel('频率')
axes[2].set_title('残差分布直方图')

plt.tight_layout()
plt.show()

In [ ]:
# ii. White 检验（异方差检验）
# H₀: 同方差; H₁: 异方差; p < 0.05 拒绝原假设 → 存在异方差
X = dfx[scol[4:]]
X_with_constant = sm.add_constant(X)
white_test = het_white(residuals, X_with_constant)

labels = ['LM统计量', 'LM p值', 'F统计量', 'F p值']
white_results = dict(zip(labels, white_test))

print("=== White 异方差检验 ===")
for key, value in white_results.items():
    print(f"{key}: {value:.4f}")
print("结论: 存在异方差（p < 0.05）" if white_test[1] < 0.05 else "结论: 不存在异方差（p ≥ 0.05）")

### 5.6 自相关检验

- **Durbin-Watson 检验**：适用于一阶自相关
- **Breusch-Godfrey 检验**：适用于高阶自相关

In [ ]:
# i. Durbin-Watson 检验
# DW ≈ 2 表示无自相关，DW < 1 或 DW > 3 表示存在自相关
dw_stat = durbin_watson(residuals)
print(f"=== Durbin-Watson 检验 ===")
print(f"DW 统计量: {dw_stat:.4f}")
print(f"结论: {'存在' if dw_stat < 1.5 or dw_stat > 2.5 else '无明显'}一阶自相关")

In [ ]:
# ii. Breusch-Godfrey 检验（滞后 4 阶）
# H₀: 无自相关; H₁: 存在自相关
bg_test = acorr_breusch_godfrey(model, nlags=4)
bg_labels = ['LM统计量', 'LM p值', 'F统计量', 'F p值']

print("=== Breusch-Godfrey 检验（滞后 4 阶）===")
for label, value in zip(bg_labels, bg_test):
    print(f"{label}: {value:.4f}")
print("结论: 存在自相关（p < 0.05）" if bg_test[1] < 0.05 else "结论: 不存在自相关（p ≥ 0.05）")

---
## 六、聚类分析

以总税收（TaxAmount）为聚类依据，使用 K-Means 算法进行区域税收分级。

### 6.1 确定最优聚类数（肘部法则）

In [ ]:
# 设置线程数以避免 KMeans 警告
os.environ['OMP_NUM_THREADS'] = '4'

# 计算 k=1 到 k=10 的 SSE（误差平方和）
sse = []
k_values = range(1, 11)

for k in k_values:
    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(dfx[["TaxAmount"]])
    sse.append(kmeans.inertia_)

# 绘制肘部图
plt.figure(figsize=(8, 6))
plt.plot(k_values, sse, 'bx-')
plt.xlabel('k 值')
plt.ylabel('误差平方和（SSE）')
plt.title('肘部法则 — 确定最优聚类数')
plt.grid(True, alpha=0.3)
plt.show()

### 6.2 拟合 K-Means 模型（k=3）

根据肘部图，选择 k=3 作为最优聚类数。

In [ ]:
# 创建并拟合 K-Means 模型
kmeans = KMeans(n_clusters=3, random_state=42)
kmeans.fit(dfx[["TaxAmount"]])
dfx['TaxAmount_Cluster'] = kmeans.labels_

print("聚类标签:", dfx['TaxAmount_Cluster'].unique())
print(f"\n各聚类样本数:\n{dfx['TaxAmount_Cluster'].value_counts().sort_index()}")

### 6.3 聚类结果可视化

查看各税收聚类下的地区数量和具体地区归属。

In [ ]:
# 各聚类下的地区数量
group_counts = dfx.groupby('TaxAmount_Cluster')['AreaName'].nunique().reset_index()

plt.figure(figsize=(10, 6))
sns.barplot(x='TaxAmount_Cluster', y='AreaName', data=group_counts, palette='viridis')
plt.title('各税收聚类下的地区数量', fontsize=14)
plt.xlabel('税收聚类（TaxAmount_Cluster）', fontsize=12)
plt.ylabel('地区数量', fontsize=12)
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

In [ ]:
# 具体地区归属
cluster_areas = dfx.groupby('TaxAmount_Cluster')['AreaName'].apply(
    lambda x: ','.join(x.unique())
).reset_index()

def format_areas(areas, max_items_per_line=4):
    """格式化地区列表，每行最多显示指定数量的地区。"""
    area_list = areas.split(', ')
    return '\n'.join(
        ', '.join(area_list[i:i + max_items_per_line])
        for i in range(0, len(area_list), max_items_per_line)
    )

cluster_areas['Formatted_Areas'] = cluster_areas['AreaName'].apply(format_areas)

# 可视化聚类-地区映射
plt.figure(figsize=(12, 8))
for i, row in cluster_areas.iterrows():
    plt.barh(y=i, width=1,
             color=sns.color_palette()[i % len(sns.color_palette())],
             label=row['TaxAmount_Cluster'])
    plt.text(x=1.1, y=i, s=row['Formatted_Areas'],
             ha='left', va='center', fontsize=10)

plt.title('税收聚类与地区归属关系', fontsize=14)
plt.xlabel('聚类-地区映射', fontsize=12)
plt.yticks(range(len(cluster_areas)), cluster_areas['TaxAmount_Cluster'])
plt.legend(title='税收聚类', bbox_to_anchor=(1, 1))
plt.xlim(0, 2)
plt.grid(axis='x', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

### 6.4 地理空间可视化

使用 GeoPandas 绘制中国省级地图，按税收聚类结果着色。
- 省级行政区划 shapefile（`.shp`）
- 海岸线 shapefile
- 九段线 shapefile

In [ ]:
# 读取 GIS 数据（路径需根据实际文件调整）
# 示例路径，请修改为你实际的 shapefile 位置
GIS_BASE = os.path.join(DATA_PATH, 'gis')

try:
    gdf = gpd.read_file(os.path.join(GIS_BASE, '省.shp'))
    coastline = gpd.read_file(os.path.join(GIS_BASE, '海岸线.shp'))
    nine_dashes = gpd.read_file(os.path.join(GIS_BASE, '九段线.shp'))
    print("GIS 数据加载成功。")
except FileNotFoundError as e:
    print(f"GIS 数据文件未找到: {e}")
    print("请将省级行政区划、海岸线、九段线的 shapefile 放入 data/gis/ 目录。")
    print("如无 GIS 数据，可跳过此步骤。")
    gdf = None

In [ ]:
# 合并聚类结果与地理数据
if gdf is not None:
    # 统一列名
    map_col = {'省': 'AreaName'}
    gdf = gdf.rename(columns=map_col)[['AreaName', 'geometry']]

    # 合并
    gdff = dfx.merge(gdf, how='left', on='AreaName')
    gdff = gpd.GeoDataFrame(gdff, geometry='geometry')

    # 绘制地图
    fig, ax = plt.subplots(figsize=(10, 6))
    cmap = ListedColormap(['#f7f7f7', '#d9d9d9', '#bdbdbd', '#969696', '#636363'])

    gdff.plot(
        column='TaxAmount_Cluster', cmap=cmap, legend=True, ax=ax,
        legend_kwds={
            'label': "TaxAmount_Cluster per Province",
            'orientation': "vertical",
            'shrink': 0.7
        },
        edgecolor='black',
        linewidth=0.5
    )

    coastline.plot(ax=ax, color='black', linewidth=0.5)
    nine_dashes.plot(ax=ax, color='red', linewidth=0.5)

    ax.set_title('Province Distribution by TaxAmount_Cluster', fontsize=14, fontweight='bold')
    ax.set_facecolor('#f0f0f0')
    ax.set_axis_off()

    plt.tight_layout()
    plt.show()
else:
    print("跳过地理空间可视化。")

---
## 七、总结

本次实验围绕产业税收展开，将产业税收数据与居民人均消费支出、人均可支配收入数据进行合并，在此基础上开展了系列探索与数据处理工作。

### 主要发现

1. **相关性分析**：热力图显示，仅有居民人均消费支出（CE_Resident）与总税收的相关系数大于 0.5，具有一般相关水平。

2. **多元线性回归**：模型的调整决定系数（Adjusted R²）为 0.542，表明具备一定的拟合优度。变量显著性检验中，其他服务消费（CE_OtherSupServices）和财产性收入（PCDI_PropertyIncome）两个变量的 P 值大于 0.1，在模型中不显著，其余变量与总产业税收之间存在一定程度的影响关系。

3. **聚类分析**：以总税收为聚类依据，通过肘部法则确定 k=3 为最优聚类数：
   - **第一组（低税收）**：海南省、青海省、宁夏回族自治区、甘肃省、西藏自治区
   - **第二组（高税收）**：北京市、上海市、浙江省、江苏省、山东省、广东省、四川省
   - **第三组（中等税收）**：重庆市、天津市、内蒙古自治区、云南省、河南省等

### 局限性

- 回归模型中变量间存在多重共线性与自相关问题，但未采取进一步处理措施
- 聚类数选择依赖肘部法则的主观判断，缺乏客观统一标准
- 未对内生性问题进行检验（如 Hausman 检验），无法全面验证高斯-马尔科夫古典假定